<a href="https://colab.research.google.com/github/Imesh-Isuranga/Statistical-Learning-e20154/blob/main/Data%20Wrangling/data_wrangling_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

## Step 1: Install Required Libraries

In [ ]:
!pip install plotly scipy scikit-learn --quiet

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## Step 3: PlottingMethods Class

In [ ]:
class PlottingMethods:
    """
    A helper class that provides modular chart generation methods.
    Each method returns an HTML-wrapped Plotly figure for flexible embedding.
    """

    def plot_bar(self, data, x_col, y_col, title='Bar Chart'):
        """
        Generate an interactive bar chart.

        Parameters:
            data (pd.DataFrame): Input DataFrame.
            x_col (str): Column name for x-axis.
            y_col (str): Column name for y-axis.
            title (str): Chart title.

        Returns:
            str: HTML string of the chart.
        """
        if data is None or data.empty:
            print('No data available for bar chart.')
            return None
        fig = px.bar(data, x=x_col, y=y_col, title=title)
        return fig.to_html(full_html=False)

    def plot_pie(self, data, names_col, values_col, title='Pie Chart'):
        """
        Generate an interactive pie chart.

        Parameters:
            data (pd.DataFrame): Input DataFrame.
            names_col (str): Column for slice labels.
            values_col (str): Column for slice values.
            title (str): Chart title.

        Returns:
            str: HTML string of the chart.
        """
        if data is None or data.empty:
            print('No data available for pie chart.')
            return None
        fig = px.pie(data, names=names_col, values=values_col, title=title)
        return fig.to_html(full_html=False)

    def plot_histogram(self, data, col, title='Histogram'):
        """
        Generate an interactive histogram.

        Parameters:
            data (pd.DataFrame): Input DataFrame.
            col (str): Column to plot.
            title (str): Chart title.

        Returns:
            str: HTML string of the chart.
        """
        if data is None or data.empty:
            print('No data available for histogram.')
            return None
        fig = px.histogram(data, x=col, title=title)
        return fig.to_html(full_html=False)


print('PlottingMethods class defined.')

## Step 4: DataInspector Class

In [ ]:
class DataInspector:
    """
    A reusable class for data ingestion, cleaning, feature engineering
    preparation, and interactive visualization of CSV datasets.
    """

    # ------------------------------------------------------------------ #
    #  1. DATA INGESTION & SANITIZATION
    # ------------------------------------------------------------------ #

    def __init__(self):
        """Initialize the DataInspector with an empty DataFrame."""
        self.df = None

    def upload_data(self):
        """
        Upload a CSV file from the local machine using Google Colab's
        file upload widget. Automatically cleans invalid values and
        corrects column data types after loading.

        Returns:
            pd.DataFrame: The loaded and sanitized DataFrame.
        """
        uploaded = files.upload()
        for filename, content in uploaded.items():
            self.df = pd.read_csv(io.BytesIO(content))
            print(f'File "{filename}" uploaded successfully.')
        self._clean_and_convert()
        return self.df

    def load_from_url(self, url):
        """
        Load a CSV dataset directly from a public URL.
        Automatically cleans invalid values and corrects types.

        Parameters:
            url (str): The direct URL to a CSV file.

        Returns:
            pd.DataFrame: The loaded and sanitized DataFrame.
        """
        self.df = pd.read_csv(url)
        print('Dataset loaded successfully.')
        self._clean_and_convert()
        return self.df

    def _clean_and_convert(self):
        """
        Internal helper: replace garbage strings with NaN, then
        force-convert columns to numeric wherever possible.
        """
        # Replace common garbage strings with NaN
        garbage = ['?', 'n/a', 'N/A', 'NA', 'NULL', 'null', '', ' ']
        self.df.replace(garbage, np.nan, inplace=True)

        # Auto-convert columns to numeric if conversion doesn't make them all-NaN
        for col in self.df.columns:
            if self.df[col].dtype == object:
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if converted.notna().sum() > 0:   # at least one valid number
                    self.df[col] = converted

        print('Invalid values replaced and column types corrected.')

    # ------------------------------------------------------------------ #
    #  2. STRUCTURAL ANALYSIS & CLEANING
    # ------------------------------------------------------------------ #

    def data_summary(self):
        """
        Display a comprehensive summary of the dataset:
        - Row and column counts
        - First 20 rows preview
        - Breakdown of numeric vs categorical columns
        - Missing value counts per column
        """
        if self.df is None:
            print('No data loaded yet.')
            return

        print('=' * 50)
        print(f'Rows   : {self.df.shape[0]}')
        print(f'Columns: {self.df.shape[1]}')
        print('=' * 50)

        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()
        print(f'Numeric columns ({len(num_cols)}): {num_cols}')
        print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
        print('=' * 50)

        print('Missing values per column:')
        print(self.df.isnull().sum())
        print('=' * 50)

        print('First 20 rows:')
        display(self.df.head(20))

    def handle_missing_values(self, column, strategy='mean', constant=None):
        """
        Impute missing values in a specified column.

        Parameters:
            column (str): The column to impute.
            strategy (str): One of 'mean', 'median', 'mode', or 'constant'.
            constant: Value used when strategy='constant'.

        Returns:
            pd.DataFrame: DataFrame with missing values filled.
        """
        if self.df is None:
            print('No data loaded.')
            return None
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return self.df

        if strategy == 'mean':
            fill_val = self.df[column].mean()
        elif strategy == 'median':
            fill_val = self.df[column].median()
        elif strategy == 'mode':
            fill_val = self.df[column].mode()[0]
        elif strategy == 'constant':
            fill_val = constant
        else:
            print('Unknown strategy. Choose mean, median, mode, or constant.')
            return self.df

        self.df[column].fillna(fill_val, inplace=True)
        print(f'Missing values in "{column}" filled with {strategy} = {fill_val}.')
        return self.df

    def remove_duplicates(self):
        """
        Remove exact duplicate rows from the dataset.

        Returns:
            pd.DataFrame: DataFrame with duplicates removed.
        """
        if self.df is None:
            print('No data loaded.')
            return None
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        after = len(self.df)
        print(f'Removed {before - after} duplicate row(s). Rows remaining: {after}.')
        return self.df

    def handle_outliers(self, column, action='flag'):
        """
        Detect outliers using the IQR method.

        IQR = Q3 - Q1
        Lower Bound = Q1 - 1.5 * IQR
        Upper Bound = Q3 + 1.5 * IQR

        Parameters:
            column (str): Numeric column to check.
            action (str): 'flag' to print outliers, 'delete' to remove those rows.

        Returns:
            pd.DataFrame: Updated DataFrame (rows removed if action='delete').
        """
        if self.df is None:
            print('No data loaded.')
            return None
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return self.df

        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outlier_mask = (self.df[column] < lower) | (self.df[column] > upper)
        outlier_count = outlier_mask.sum()
        print(f'Outliers detected in "{column}": {outlier_count}')
        print(f'Bounds -> Lower: {lower:.2f}, Upper: {upper:.2f}')

        if action == 'flag':
            display(self.df[outlier_mask])
        elif action == 'delete':
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f'{outlier_count} outlier row(s) removed.')
        return self.df

    def delete_rows(self):
        """
        Interactively delete rows by index.
        Prompts the user to enter comma-separated row indices to remove.

        Returns:
            pd.DataFrame: DataFrame with specified rows deleted.
        """
        if self.df is None:
            print('No data loaded.')
            return None
        user_input = input('Enter row indices to delete (comma-separated): ')
        indices = [int(i.strip()) for i in user_input.split(',') if i.strip().isdigit()]
        self.df.drop(index=indices, inplace=True, errors='ignore')
        self.df.reset_index(drop=True, inplace=True)
        print(f'Deleted rows: {indices}')
        return self.df

    def delete_columns(self):
        """
        Interactively delete columns by name.
        Prompts the user to enter comma-separated column names to remove.

        Returns:
            pd.DataFrame: DataFrame with specified columns deleted.
        """
        if self.df is None:
            print('No data loaded.')
            return None
        user_input = input('Enter column names to delete (comma-separated): ')
        cols = [c.strip() for c in user_input.split(',')]
        self.df.drop(columns=cols, inplace=True, errors='ignore')
        print(f'Deleted columns: {cols}')
        return self.df

    # ------------------------------------------------------------------ #
    #  3. FEATURE ENGINEERING PREPARATION (NORMALIZATION)
    # ------------------------------------------------------------------ #

    def extract_normalized_numeric_data(self, method='minmax'):
        """
        Scale all numeric columns using the chosen method.

        Parameters:
            method (str): 'minmax', 'standard' (Z-score), or 'robust' (IQR-based).

        Returns:
            pd.DataFrame: Scaled numeric DataFrame.
        """
        if self.df is None:
            print('No data loaded.')
            return None

        num_df = self.df.select_dtypes(include=np.number).copy()

        scalers = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }
        if method not in scalers:
            print('Unknown method. Use minmax, standard, or robust.')
            return None

        scaler = scalers[method]
        scaled = scaler.fit_transform(num_df)
        result = pd.DataFrame(scaled, columns=num_df.columns)
        print(f'Numeric data scaled using "{method}" method.')
        return result

    def extract_normalized_categorical_data(self, method='onehot'):
        """
        Encode all categorical columns using the chosen method.

        Parameters:
            method (str): 'onehot', 'ordinal', or 'uniform' (scaled 0-1).

        Returns:
            pd.DataFrame: Encoded categorical DataFrame.
        """
        if self.df is None:
            print('No data loaded.')
            return None

        cat_df = self.df.select_dtypes(exclude=np.number).copy()
        if cat_df.empty:
            print('No categorical columns found.')
            return pd.DataFrame()

        # Fill NaN with 'Unknown' before encoding
        cat_df.fillna('Unknown', inplace=True)

        if method == 'onehot':
            enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = enc.fit_transform(cat_df)
            col_names = enc.get_feature_names_out(cat_df.columns)
            result = pd.DataFrame(encoded, columns=col_names)

        elif method == 'ordinal':
            enc = OrdinalEncoder()
            encoded = enc.fit_transform(cat_df)
            result = pd.DataFrame(encoded, columns=cat_df.columns)

        elif method == 'uniform':
            # Map each category to a value between 0 and 1
            result = cat_df.copy()
            for col in result.columns:
                cats = result[col].unique()
                mapping = {c: i / (len(cats) - 1) if len(cats) > 1 else 0
                           for i, c in enumerate(cats)}
                result[col] = result[col].map(mapping)
        else:
            print('Unknown method. Use onehot, ordinal, or uniform.')
            return None

        print(f'Categorical data encoded using "{method}" method.')
        return result

    def get_merged_dataset(self, numeric_method='minmax', cat_method='ordinal'):
        """
        Combine scaled numeric data and encoded categorical data into one DataFrame.

        Parameters:
            numeric_method (str): Scaling method for numeric columns.
            cat_method (str): Encoding method for categorical columns.

        Returns:
            pd.DataFrame: Merged DataFrame of processed features.
        """
        num_df = self.extract_normalized_numeric_data(method=numeric_method)
        cat_df = self.extract_normalized_categorical_data(method=cat_method)

        if num_df is None:
            return cat_df
        if cat_df is None or cat_df.empty:
            return num_df

        merged = pd.concat([num_df.reset_index(drop=True),
                            cat_df.reset_index(drop=True)], axis=1)
        print('Merged dataset created successfully.')
        return merged

    # ------------------------------------------------------------------ #
    #  4. ADVANCED INTERACTIVE VISUALIZATION
    # ------------------------------------------------------------------ #

    def plot_univariate(self, column):
        """
        Generate a 3-panel subplot for a numeric column:
        Panel 1 - Horizontal Violin / Box plot
        Panel 2 - Scatter plot (Index vs Value)
        Panel 3 - Histogram

        Parameters:
            column (str): Numeric column to visualize.
        """
        if self.df is None or column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return

        series = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('Violin / Box', 'Scatter (Index vs Value)', 'Histogram')
        )

        # Panel 1 – Violin
        fig.add_trace(
            go.Violin(y=series, name=column, box_visible=True, meanline_visible=True),
            row=1, col=1
        )
        # Panel 2 – Scatter
        fig.add_trace(
            go.Scatter(x=series.index, y=series, mode='markers', name=column),
            row=1, col=2
        )
        # Panel 3 – Histogram
        fig.add_trace(
            go.Histogram(x=series, name=column),
            row=1, col=3
        )

        fig.update_layout(title_text=f'Univariate Analysis: {column}', showlegend=False)
        fig.show()

    def plot_relationship(self, col_x, col_y):
        """
        Auto-detect column types and plot the appropriate chart:
        - Numeric vs Numeric  -> Scatter with OLS trendline
        - Categorical vs Numeric -> Box plot with data points
        - Categorical vs Categorical -> Grouped bar chart

        Parameters:
            col_x (str): First column name.
            col_y (str): Second column name.
        """
        if self.df is None:
            print('No data loaded.')
            return

        x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        if x_num and y_num:
            # Numeric – Numeric: scatter with OLS trendline
            fig = px.scatter(self.df, x=col_x, y=col_y,
                             trendline='ols',
                             title=f'{col_x} vs {col_y} (OLS Trendline)')

        elif not x_num and y_num:
            # Categorical – Numeric: box plot
            fig = px.box(self.df, x=col_x, y=col_y,
                         points='all',
                         title=f'{col_x} vs {col_y} (Box Plot)')

        elif x_num and not y_num:
            # Numeric – Categorical: flip roles
            fig = px.box(self.df, x=col_y, y=col_x,
                         points='all',
                         title=f'{col_y} vs {col_x} (Box Plot)')

        else:
            # Categorical – Categorical: grouped bar chart
            ct = pd.crosstab(self.df[col_x], self.df[col_y]).reset_index()
            ct_melted = ct.melt(id_vars=col_x, var_name=col_y, value_name='Count')
            fig = px.bar(ct_melted, x=col_x, y='Count', color=col_y,
                         barmode='group',
                         title=f'{col_x} vs {col_y} (Grouped Bar)')

        fig.show()

    def plot_categorical_frequency(self, column):
        """
        Plot a bar chart showing raw counts and percentage labels
        for a categorical column.

        Parameters:
            column (str): Categorical column to visualize.
        """
        if self.df is None or column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        counts['Percentage'] = (counts['Count'] / counts['Count'].sum() * 100).round(1)
        counts['Label'] = counts['Percentage'].astype(str) + '%'

        fig = px.bar(counts, x=column, y='Count',
                     text='Label',
                     title=f'Frequency of {column}')
        fig.update_traces(textposition='outside')
        fig.show()

    # ------------------------------------------------------------------ #
    #  5. DEEP STATISTICAL INSIGHTS – UNIFIED ASSOCIATION HEATMAP
    # ------------------------------------------------------------------ #

    def plot_all_associations_heatmap(self):
        """
        Build and display a unified association heatmap across all column types:
        - Numeric vs Numeric   : Pearson r
        - Categorical vs Categorical : Cramer's V
        - Numeric vs Categorical     : Point-Biserial correlation (or Eta via ANOVA)
        """
        if self.df is None:
            print('No data loaded.')
            return

        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)

        def cramers_v(x, y):
            """Compute Cramer's V statistic for two categorical columns."""
            ct = pd.crosstab(x, y)
            chi2 = stats.chi2_contingency(ct)[0]
            n_obs = ct.sum().sum()
            min_dim = min(ct.shape) - 1
            if min_dim == 0 or n_obs == 0:
                return 0.0
            return np.sqrt(chi2 / (n_obs * min_dim))

        def eta_squared(numeric_col, cat_col):
            """Compute Eta-squared between a numeric and categorical column."""
            groups = [group.dropna().values
                      for _, group in numeric_col.groupby(cat_col)]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) < 2:
                return 0.0
            f_stat, p = stats.f_oneway(*groups)
            # Return magnitude only (0 to 1)
            ss_between = sum(len(g) * (g.mean() - numeric_col.mean())**2 for g in groups)
            ss_total = sum((numeric_col - numeric_col.mean())**2)
            return ss_between / ss_total if ss_total != 0 else 0.0

        for i, c1 in enumerate(cols):
            for j, c2 in enumerate(cols):
                if i == j:
                    matrix.iloc[i, j] = 1.0
                    continue
                s1 = self.df[c1].dropna()
                s2 = self.df[c2].dropna()
                common_idx = s1.index.intersection(s2.index)
                s1, s2 = s1[common_idx], s2[common_idx]

                num1 = pd.api.types.is_numeric_dtype(s1)
                num2 = pd.api.types.is_numeric_dtype(s2)

                if num1 and num2:
                    r, _ = stats.pearsonr(s1, s2)
                    matrix.iloc[i, j] = round(r, 3)
                elif not num1 and not num2:
                    matrix.iloc[i, j] = round(cramers_v(s1, s2), 3)
                else:
                    # Numeric vs Categorical
                    num_col = s1 if num1 else s2
                    cat_col = s2 if num1 else s1
                    eta = eta_squared(num_col, cat_col)
                    matrix.iloc[i, j] = round(eta, 3)

        fig = px.imshow(
            matrix,
            text_auto=True,
            color_continuous_scale='RdBu_r',
            zmin=-1, zmax=1,
            title='Unified Association Heatmap (Pearson r | Cramér\'s V | Eta²)'
        )
        fig.update_layout(width=900, height=700)
        fig.show()


print('DataInspector class defined successfully.')

## Step 5: Full Demonstration – Titanic Dataset
### Upload → Impute → Normalize → Visualize Associations

In [ ]:
# --- Load Dataset from URL ---
inspector = DataInspector()
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = inspector.load_from_url(url)
df.head()

In [ ]:
# --- Dataset Summary ---
inspector.data_summary()

In [ ]:
# --- Remove Duplicates ---
inspector.remove_duplicates()

In [ ]:
# --- Handle Missing Values ---
# Age column: fill with median (age is skewed)
inspector.handle_missing_values('Age', strategy='median')

# Embarked column: fill with mode (most common port)
inspector.handle_missing_values('Embarked', strategy='mode')

# Fare column: fill with mean
inspector.handle_missing_values('Fare', strategy='mean')

In [ ]:
# --- Detect Outliers in Fare column ---
inspector.handle_outliers('Fare', action='flag')

In [ ]:
# --- Univariate Visualization for Age ---
inspector.plot_univariate('Age')

In [ ]:
# --- Relationship Plots ---
# Numeric vs Numeric: Age vs Fare
inspector.plot_relationship('Age', 'Fare')

# Categorical vs Numeric: Sex vs Fare
inspector.plot_relationship('Sex', 'Fare')

# Categorical vs Categorical: Sex vs Survived
inspector.plot_relationship('Sex', 'Survived')

In [ ]:
# --- Categorical Frequency Chart ---
inspector.plot_categorical_frequency('Pclass')
inspector.plot_categorical_frequency('Sex')

In [ ]:
# --- Normalize Numeric Data ---
scaled_df = inspector.extract_normalized_numeric_data(method='minmax')
print(scaled_df.head())

In [ ]:
# --- Encode Categorical Data ---
encoded_df = inspector.extract_normalized_categorical_data(method='ordinal')
print(encoded_df.head())

In [ ]:
# --- Merged Final Dataset ---
final_df = inspector.get_merged_dataset(numeric_method='minmax', cat_method='ordinal')
print(f'Final merged dataset shape: {final_df.shape}')
final_df.head()

In [ ]:
# --- Unified Association Heatmap ---
# Using a subset of key columns for clarity
inspector.df = inspector.df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
inspector.plot_all_associations_heatmap()

## Step 6: PlottingMethods Demo

In [ ]:
# --- PlottingMethods Demo ---
plotter = PlottingMethods()

# Reload fresh data for plotting demo
demo_df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Bar chart: Survival count by class
class_survival = demo_df.groupby('Pclass')['Survived'].sum().reset_index()
html_bar = plotter.plot_bar(class_survival, x_col='Pclass', y_col='Survived',
                            title='Survivors by Passenger Class')

# Pie chart: Gender distribution
gender_counts = demo_df['Sex'].value_counts().reset_index()
gender_counts.columns = ['Sex', 'Count']
html_pie = plotter.plot_pie(gender_counts, names_col='Sex', values_col='Count',
                            title='Gender Distribution')

# Histogram
html_hist = plotter.plot_histogram(demo_df, col='Age', title='Age Distribution')

# Render the HTML outputs
from IPython.display import HTML, display
display(HTML(html_bar))
display(HTML(html_pie))
display(HTML(html_hist))